# Approach 2 — Clustering-Based Archetype Classification

## What this approach does

Clustering lets the **data itself decide** how practices naturally group together, without
you specifying the boundaries in advance. The algorithm finds practices that are similar
to each other across multiple dimensions simultaneously and groups them into clusters.
Those clusters are then mapped onto the framework labels (Small/Foundation, NHS Led, etc.)
by comparing each cluster's centroid to the expected ordering of the labels.

We run **two independent K-Means models** — one for size dimensions and one for model
dimensions — rather than a single model on all features. This mirrors the framework's
own two-axis structure and keeps the results interpretable.

## Why use it?

| Strength | Detail |
|----------|--------|
| Data-driven | Boundaries reflect actual practice behaviour, not assumed thresholds |
| Multi-dimensional | Groups on several features at once — no need to pick a single signal |
| Discovers nuance | May reveal natural groupings that differ from the framework's assumptions |
| Validates rules | Comparing clustering to rules labels shows where the rules are well-calibrated |

## Limitations

- Results can change slightly if the data distribution shifts (K-Means is sensitive to scale)
- The number of clusters (k=4) is chosen to match the framework — the algorithm does not
  automatically determine the right k
- Labels require post-hoc interpretation: we have to decide which cluster is "NHS Led"
  after the fact by inspecting centroid values
- K-Means assumes spherical clusters and can struggle with very skewed distributions

## How K-Means works

1. Place k centroids randomly in the feature space
2. Assign every practice to its nearest centroid
3. Recompute each centroid as the mean of its assigned practices
4. Repeat steps 2–3 until assignments stop changing

Because distance is sensitive to scale, all features are standardised with
`StandardScaler` (zero mean, unit variance) before fitting.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

warnings.filterwarnings('ignore')

BLUE   = '#4C72B0'
ORANGE = '#DD8452'
GREEN  = '#55A868'
RED    = '#C44E52'
RANDOM_STATE = 42

sns.set_theme(style='whitegrid', palette='Set2', font_scale=1.05)
plt.rcParams.update({'figure.dpi': 120, 'axes.spines.top': False, 'axes.spines.right': False})

SIZE_ORDER  = ['Small/Foundation', 'Medium/Core', 'Large/Advanced', 'Flagship']
MODEL_ORDER = ['NHS Led', 'Balanced Mixed', 'Private Led Mixed', 'Specialist/Referral Hub']
NA_ZONE_PAIRS = {('NHS Led', 'Large/Advanced'), ('NHS Led', 'Flagship')}

df = pd.read_csv('master.csv')
print(f'Loaded {len(df):,} practices')

---
## Feature engineering

In [ ]:
df['nhs_income_est']   = df['nhsincome'].fillna(0)
df['private_income']   = df['privateincome'].fillna(0)
df['total_income_est'] = df['private_income'] + df['nhs_income_est']

df['nhs_share'] = np.where(
    df['total_income_est'] > 0,
    df['nhs_income_est'] / df['total_income_est'],
    np.nan
)

df['items_per_surgery']        = df['nooftreatmentitems'] / df['numberofsurgeries'].replace(0, np.nan)
df['private_income_per_chair'] = df['private_income'] / df['numberofchairs'].replace(0, np.nan)

print('Features ready')

---
## Elbow analysis — choosing k

The **elbow chart** plots inertia (sum of squared distances from each point to its cluster
centroid) as a function of k. A sharp "elbow" at k=4 would confirm that four clusters
captures the most natural structure without overfitting.

We run this separately for the size and model feature sets. The framework prescribes
k=4 for both, but it is good practice to verify this on your actual data.

In [ ]:
scaler = StandardScaler()

size_features  = ['numberofsurgeries', 'unique_staff_ids', 'contractualhours_dentist']
model_features = ['nhs_share', 'private_income', 'nhs_income_est',
                  'items_per_surgery', 'position_hygienist', 'private_income_per_chair']

X_size  = scaler.fit_transform(df[size_features].fillna(0))
X_model = scaler.fit_transform(df[model_features].fillna(0))

k_range = range(2, 9)
inertia_size  = [KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10).fit(X_size).inertia_  for k in k_range]
inertia_model = [KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10).fit(X_model).inertia_ for k in k_range]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle('Elbow Analysis — Selecting k for Each Dimension', fontweight='bold')

for ax, inertia, title, color in [
    (axes[0], inertia_size,  'Size features', BLUE),
    (axes[1], inertia_model, 'Model features', ORANGE),
]:
    ax.plot(list(k_range), inertia, marker='o', color=color, lw=2)
    ax.axvline(4, color=RED, lw=1.5, ls='--', label='k = 4 (framework)')
    ax.set_xlabel('Number of clusters (k)')
    ax.set_ylabel('Inertia')
    ax.set_title(title)
    ax.legend()

plt.tight_layout()
plt.show()

---
## Fitting the clusters

### Feature selection rationale

**Size clustering** uses three capacity/workforce features:
- `numberofsurgeries` — physical footprint
- `unique_staff_ids` — workforce scale
- `contractualhours_dentist` — clinical hour commitment

These three together characterise how *big* a practice is without introducing income
signals, which belong to the model dimension.

**Model clustering** uses six income and activity features:
- `nhs_share` — primary NHS/private orientation signal
- `private_income` and `nhs_income_est` — absolute revenue by stream
- `items_per_surgery` — activity intensity (a proxy for patient demand type)
- `position_hygienist` — indicator of elective/specialist services
- `private_income_per_chair` — private revenue productivity (specialist signal)

### Mapping clusters to labels

After fitting, each cluster is assigned a label by ranking its centroid value on the
most discriminating feature:
- Size clusters ranked by mean `numberofsurgeries` (ascending: Small → Flagship)
- Model clusters ranked by mean `nhs_share` (descending: NHS Led → Specialist)

In [ ]:
def fit_and_label(X, df_ref, cluster_col, ordering_col, labels, ascending=True):
    km = KMeans(n_clusters=4, random_state=RANDOM_STATE, n_init=10)
    df_ref = df_ref.copy()
    df_ref[cluster_col] = km.fit_predict(X)
    centroid_means = (
        df_ref.groupby(cluster_col)[ordering_col]
        .mean()
        .sort_values(ascending=ascending)
    )
    mapping = {cid: labels[rank] for rank, cid in enumerate(centroid_means.index)}
    return df_ref[cluster_col], df_ref[cluster_col].map(mapping), km

# Re-fit scalers for the final models
scaler_size  = StandardScaler()
scaler_model = StandardScaler()
X_size  = scaler_size.fit_transform(df[size_features].fillna(0))
X_model = scaler_model.fit_transform(df[model_features].fillna(0))

df['cluster_size_id'],  df['archetype_size'],  km_size  = fit_and_label(
    X_size,  df, 'cluster_size_id',  'numberofsurgeries', SIZE_ORDER,  ascending=True)
df['cluster_model_id'], df['archetype_model'], km_model = fit_and_label(
    X_model, df, 'cluster_model_id', 'nhs_share',         MODEL_ORDER, ascending=False)

# N/A zone enforcement
for model_val, size_val in NA_ZONE_PAIRS:
    mask = (df['archetype_model'] == model_val) & (df['archetype_size'] == size_val)
    n = mask.sum()
    df.loc[mask, 'archetype_model'] = 'Balanced Mixed'
    if n:
        print(f'N/A zone: reclassified {n} [{model_val} x {size_val}] -> Balanced Mixed')

df['archetype_clust'] = df['archetype_size'] + ' | ' + df['archetype_model']

print(f'\nSize cluster distribution:')
print(df['archetype_size'].value_counts().reindex(SIZE_ORDER).to_string())
print(f'\nModel cluster distribution:')
print(df['archetype_model'].value_counts().reindex(MODEL_ORDER).to_string())

---
## Cluster profiles

The radar/parallel-coordinates view below shows how each cluster differs across the
features used in clustering. Distinct separation between clusters on multiple features
confirms the clustering is meaningful.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
fig.suptitle('Cluster Centroids — Normalised Feature Values', fontsize=13, fontweight='bold')

palette = sns.color_palette('Set2', 4)

for ax, X_sc, km, labels, features, title in [
    (axes[0], X_size,  km_size,  SIZE_ORDER,  size_features,  'Size clusters'),
    (axes[1], X_model, km_model, MODEL_ORDER, model_features, 'Model clusters'),
]:
    # Map raw cluster ids to labels via centroid rank
    centroid_means = pd.DataFrame(km.cluster_centers_, columns=features)
    ordering_col   = features[0]   # numberofsurgeries or nhs_share
    ascending      = True if ax == axes[0] else False
    ranked_ids     = centroid_means[ordering_col].sort_values(ascending=ascending).index
    label_map      = {cid: labels[rank] for rank, cid in enumerate(ranked_ids)}

    x = np.arange(len(features))
    for raw_id, label in label_map.items():
        rank  = list(label_map.values()).index(label)
        color = palette[rank]
        ax.plot(x, km.cluster_centers_[raw_id], marker='o', lw=2, color=color, label=label)

    ax.set_xticks(x)
    ax.set_xticklabels([f.replace('_', '\n') for f in features], fontsize=8)
    ax.set_ylabel('Standardised value')
    ax.set_title(title)
    ax.axhline(0, color='grey', lw=0.8, ls='--')
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

---
## Cluster scatter visualisations — practices in feature space

The centroid profiles above show the *average* of each cluster. The scatter plots
below show **every practice as an individual dot**, coloured by its cluster label,
plotted on the actual feature axes used for clustering.

This answers questions the centroid view cannot:
- **How separated are the clusters?** Clear gaps mean a clean, stable solution;
  overlap means the boundary is genuinely ambiguous in that feature combination.
- **Are there outliers?** A practice far from its cluster centre may have been
  mis-assigned or may be a genuinely unusual operation.
- **Which feature pair best separates the clusters?** The pair where clusters
  form the cleanest separate clouds is the most discriminating dimension.

> **Reading the charts**: axes are raw, unscaled feature values — no
> transformation has been applied.  Stars (★) mark cluster centroids.
> The `StandardScaler` was only used internally by K-Means and is
> not reflected in the axis values shown here.

### Size clusters — scatter grid

Three features drive the size clustering: `numberofsurgeries`, `unique_staff_ids`
and `contractualhours_dentist`. With three features there are three distinct 2D
projections — each pair of charts shows a different slice through the same
cluster assignment.

In [ ]:
SIZE_PALETTE = {
    'Small/Foundation': '#74add1',
    'Medium/Core':      '#4575b4',
    'Large/Advanced':   '#d73027',
    'Flagship':         '#a50026',
}

# Feature pairs for size
size_pairs = [
    ('numberofsurgeries',    'unique_staff_ids',          'Surgeries',     'Staff headcount'),
    ('numberofsurgeries',    'contractualhours_dentist',   'Surgeries',     'Dentist contracted hrs'),
    ('unique_staff_ids',     'contractualhours_dentist',   'Staff headcount','Dentist contracted hrs'),
]

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Size Clusters — All Feature Pairs (raw values)', fontsize=13, fontweight='bold')

# Inverse-transform centroids to raw feature scale
centroids_size_raw = scaler_size.inverse_transform(km_size.cluster_centers_)
centroid_df_size   = pd.DataFrame(centroids_size_raw, columns=size_features)

# Map raw cluster id to label
ordering = centroid_df_size['numberofsurgeries'].sort_values(ascending=True)
size_id_to_label = {cid: SIZE_ORDER[rank] for rank, cid in enumerate(ordering.index)}

for ax, (xcol, ycol, xlabel, ylabel) in zip(axes, size_pairs):
    for label in SIZE_ORDER:
        mask  = df['archetype_size'] == label
        color = SIZE_PALETTE[label]
        ax.scatter(df.loc[mask, xcol], df.loc[mask, ycol],
                   color=color, alpha=0.45, s=28, label=label, edgecolors='none')

    # Plot centroids (inverse-transformed)
    for raw_id, label in size_id_to_label.items():
        cx = centroids_size_raw[raw_id, size_features.index(xcol)]
        cy = centroids_size_raw[raw_id, size_features.index(ycol)]
        ax.scatter(cx, cy, color=SIZE_PALETTE[label],
                   s=220, marker='*', edgecolors='black', linewidths=0.8, zorder=5)

    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.set_title(f'{xlabel} vs {ylabel}')

# One shared legend
handles = [plt.scatter([], [], color=SIZE_PALETTE[l], label=l, s=40) for l in SIZE_ORDER]
axes[0].legend(handles=handles, fontsize=8, loc='upper left')

plt.tight_layout()
plt.show()

#### Interpretation — size scatter

- **Well-separated clusters** (clear horizontal/vertical bands) confirm that the
  K-Means solution reflects genuine structure in the data, not arbitrary partition.
- The `Surgeries` axis is the dominant separator — this is expected since it is also
  the ordering feature used to map raw cluster IDs to labels.
- `Staff headcount` adds a second dimension: Flagship practices sit apart from
  Large/Advanced even at the same surgery count because of their higher workforce volume.
- **Practices near a boundary** (e.g. a Medium/Core practice at the top of its cloud)
  are the most likely mis-assignments — compare their rules-based label against their
  clustering label in the agreement section below.

### Model clusters — scatter grid

Six features drive the model clustering. Rather than showing all 15 possible pairs,
the six most discriminating combinations are shown — chosen to reveal the
NHS/private axis, the income scale axis, and the specialist axis independently.

In [ ]:
MODEL_PALETTE = {
    'NHS Led':                '#1a9850',
    'Balanced Mixed':          '#fee08b',
    'Private Led Mixed':       '#f46d43',
    'Specialist/Referral Hub': '#9e0142',
}

# Six most informative model feature pairs
model_pairs = [
    ('nhs_share',              'private_income',         'NHS share of income',     'Private income (£)'),
    ('nhs_share',              'private_income_per_chair','NHS share of income',     'Private income per chair (£)'),
    ('nhs_share',              'position_hygienist',      'NHS share of income',     'Hygienists'),
    ('private_income',         'nhs_income_est',          'Private income (£)',       'NHS income est. (£)'),
    ('private_income_per_chair','items_per_surgery',      'Private income per chair (£)','Items per surgery'),
    ('private_income',         'items_per_surgery',       'Private income (£)',       'Items per surgery'),
]

fig, axes = plt.subplots(2, 3, figsize=(17, 10))
fig.suptitle('Model Clusters — Key Feature Pairs (raw values)', fontsize=13, fontweight='bold')

# Inverse-transform model centroids
centroids_model_raw = scaler_model.inverse_transform(km_model.cluster_centers_)
centroid_df_model   = pd.DataFrame(centroids_model_raw, columns=model_features)

ordering_model  = centroid_df_model['nhs_share'].sort_values(ascending=False)
model_id_to_label = {cid: MODEL_ORDER[rank] for rank, cid in enumerate(ordering_model.index)}

for ax, (xcol, ycol, xlabel, ylabel) in zip(axes.flat, model_pairs):
    for label in MODEL_ORDER:
        mask  = df['archetype_model'] == label
        color = MODEL_PALETTE[label]
        ax.scatter(df.loc[mask, xcol], df.loc[mask, ycol],
                   color=color, alpha=0.45, s=28, label=label, edgecolors='none')

    # Centroids
    for raw_id, label in model_id_to_label.items():
        feat_idx_x = model_features.index(xcol)
        feat_idx_y = model_features.index(ycol)
        cx = centroids_model_raw[raw_id, feat_idx_x]
        cy = centroids_model_raw[raw_id, feat_idx_y]
        ax.scatter(cx, cy, color=MODEL_PALETTE[label],
                   s=220, marker='*', edgecolors='black', linewidths=0.8, zorder=5)

    # Format £ axes
    for axis_obj, col in [(ax.xaxis, xcol), (ax.yaxis, ycol)]:
        if 'income' in col:
            axis_obj.set_major_formatter(
                mticker.FuncFormatter(lambda x, _: f'£{x/1000:.0f}k' if x >= 1000 else f'£{x:.0f}'))

    ax.set_xlabel(xlabel, fontsize=9)
    ax.set_ylabel(ylabel, fontsize=9)
    ax.set_title(f'{xlabel.split("(")[0].strip()} vs {ylabel.split("(")[0].strip()}', fontsize=9)

handles = [plt.scatter([], [], color=MODEL_PALETTE[l], label=l, s=40) for l in MODEL_ORDER]
axes[0, 0].legend(handles=handles, fontsize=8, loc='upper right')

plt.tight_layout()
plt.show()

#### Interpretation — model scatter

- **NHS share vs Private income (top-left)**: the most readable summary. NHS Led
  practices cluster top-left (high NHS share, lower private income); Specialist/Referral
  Hub sits bottom-right (low NHS share, higher private); Balanced Mixed occupies the
  middle diagonal.
- **NHS share vs Hygienists (top-right)**: hygienist presence is a strong marker for
  Specialist/Referral Hub — clusters that sit at higher hygienist counts with lower NHS
  share are the clearest specialist signal.
- **Private income vs NHS income (middle)**: shows the income decomposition trade-off.
  NHS Led practices lie along the NHS axis; private practices along the private axis.
  Balanced Mixed sits on the diagonal.
- **Private income per chair vs Items per surgery (bottom-middle)**: a high-yield
  (income per chair) but lower-throughput (items per surgery) cluster is a Specialist
  signature — fewer but higher-value procedures. A high-throughput but lower-yield
  cluster is a high-volume NHS generalist.

### Parallel coordinates — all features simultaneously

A parallel coordinates chart places each feature on its own vertical axis. Each practice
is drawn as a line that passes through all axes at its feature values.  Lines are coloured
by cluster.  This reveals:

- **Where clusters diverge**: axes where lines fan out into colour-distinct bands
- **Where clusters overlap**: axes where all colours mix together (low discrimination)
- **Correlated features**: axes where lines run roughly parallel across many practices
  (the features move together)

In [ ]:
from pandas.plotting import parallel_coordinates

# ── Size parallel coordinates ─────────────────────────────────────────────────
pc_size = df[size_features + ['archetype_size']].copy()
# Min-max normalise each feature to 0-1 so all axes share the same range
for col in size_features:
    lo, hi = pc_size[col].min(), pc_size[col].max()
    pc_size[col] = (pc_size[col] - lo) / (hi - lo + 1e-9)

# Add size-band mean lines for clarity (will overlay)
fig, ax = plt.subplots(figsize=(9, 5))
fig.suptitle('Size Clusters — Parallel Coordinates (min-max scaled)', fontsize=13, fontweight='bold')

for label, color in SIZE_PALETTE.items():
    subset = pc_size[pc_size['archetype_size'] == label].drop(columns='archetype_size')
    # Draw individual practices (thin, transparent)
    for _, row in subset.iterrows():
        ax.plot(range(len(size_features)), row.values, color=color, alpha=0.08, lw=0.8)
    # Draw cluster mean (bold)
    ax.plot(range(len(size_features)), subset.mean().values,
            color=color, lw=3.0, alpha=0.95, label=label)

ax.set_xticks(range(len(size_features)))
ax.set_xticklabels(['Surgeries', 'Staff
headcount', 'Dentist
contracted hrs'], fontsize=10)
ax.set_ylabel('Normalised value (0 = portfolio min, 1 = max)')
ax.set_yticks([0, 0.25, 0.5, 0.75, 1.0])
ax.legend(title='Size band', fontsize=9)
plt.tight_layout()
plt.show()

# ── Model parallel coordinates ────────────────────────────────────────────────
pc_model = df[model_features + ['archetype_model']].copy()
for col in model_features:
    lo, hi = pc_model[col].min(), pc_model[col].max()
    pc_model[col] = (pc_model[col] - lo) / (hi - lo + 1e-9)

model_labels_display = ['NHS
share', 'Private
income', 'NHS
income', 
                         'Items/
surgery', 'Hygienists', 'Priv income
per chair']

fig, ax = plt.subplots(figsize=(12, 5))
fig.suptitle('Model Clusters — Parallel Coordinates (min-max scaled)', fontsize=13, fontweight='bold')

for label, color in MODEL_PALETTE.items():
    subset = pc_model[pc_model['archetype_model'] == label].drop(columns='archetype_model')
    for _, row in subset.iterrows():
        ax.plot(range(len(model_features)), row.values, color=color, alpha=0.07, lw=0.8)
    ax.plot(range(len(model_features)), subset.mean().values,
            color=color, lw=3.0, alpha=0.95, label=label)

ax.set_xticks(range(len(model_features)))
ax.set_xticklabels(model_labels_display, fontsize=9)
ax.set_ylabel('Normalised value (0 = portfolio min, 1 = max)')
ax.set_yticks([0, 0.25, 0.5, 0.75, 1.0])
ax.legend(title='Model band', fontsize=9, loc='upper right')
plt.tight_layout()
plt.show()

#### Interpretation — parallel coordinates

**Size clusters**: the three axes are positively correlated (more surgeries → more staff
→ more dentist hours), so lines run broadly parallel across clusters.  The chart confirms
that K-Means found a clean ordinal separation: Small/Foundation lines sit low on all
three axes; Flagship lines sit high.  The **spread within each cluster** (the width of
each colour band at each axis) shows how much variance remains *within* a tier.

**Model clusters**: the NHS share and private income axes should show the clearest
separation.
- **NHS Led** (green): high on `NHS share`, lower on `Private income`
- **Private Led Mixed** (orange): the reverse — low NHS share, high private income
- **Balanced Mixed** (yellow): mid-range on both income axes, moderate hygienist count
- **Specialist/Referral Hub** (dark red): low NHS share, highest private income per
  chair, and the highest hygienist count — the specialist signature is visible as a
  spike on the right-hand axes

**Where lines cross** between two adjacent axes: the two features are *inversely*
correlated for practices in the crossing cluster — worth understanding (e.g. a practice
with high NHS share but low NHS income may have an under-delivered contract).

---
## 4 × 4 Matrix

In [ ]:
ct = pd.crosstab(df['archetype_size'], df['archetype_model']).reindex(
    index=SIZE_ORDER, columns=MODEL_ORDER, fill_value=0
)

fig, ax = plt.subplots(figsize=(9, 5))
sns.heatmap(ct, ax=ax, annot=True, fmt='d', cmap='Blues',
            linewidths=0.5, linecolor='white', cbar_kws={'shrink': 0.7})
ax.set_title('Clustering — Practice Count Matrix', fontsize=13, fontweight='bold')
ax.set_xlabel('Model')
ax.set_ylabel('Size')
ax.tick_params(axis='x', rotation=20)
ax.tick_params(axis='y', rotation=0)
for model_val, size_val in NA_ZONE_PAIRS:
    ri, ci = SIZE_ORDER.index(size_val), MODEL_ORDER.index(model_val)
    ax.add_patch(plt.Rectangle((ci, ri), 1, 1, fill=True, color='#d0d0d0', zorder=3, alpha=0.8))
    ax.text(ci + 0.5, ri + 0.5, 'N/A', ha='center', va='center', fontsize=9, color='grey', zorder=4)

plt.tight_layout()
plt.show()

print(ct.to_string())

---
## Agreement with rules-based labels

Comparing the clustering output to the rules-based labels (from `archetypes_rules.csv`)
reveals where the two approaches agree and where they diverge.

- **High agreement** on a cell means the rules thresholds are well-calibrated to the data
- **Disagreement** often highlights practices near a boundary — these are candidates for
  closer manual review

> If `archetypes_rules.csv` does not exist yet, run `01_rules_based.ipynb` first.

In [ ]:
import os

if os.path.exists('archetypes_rules.csv'):
    rules = pd.read_csv('archetypes_rules.csv')[['practicekey', 'archetype_size', 'archetype_model']]
    rules.columns = ['practicekey', 'rules_size', 'rules_model']
    merged = df.merge(rules, on='practicekey', how='left')

    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    fig.suptitle('Rules vs Clustering Agreement', fontsize=13, fontweight='bold')

    size_agree  = pd.crosstab(merged['rules_size'],  merged['archetype_size']).reindex(
        index=SIZE_ORDER, columns=SIZE_ORDER, fill_value=0)
    model_agree = pd.crosstab(merged['rules_model'], merged['archetype_model']).reindex(
        index=MODEL_ORDER, columns=MODEL_ORDER, fill_value=0)

    sns.heatmap(size_agree,  ax=axes[0], annot=True, fmt='d', cmap='Greens',
                linewidths=0.5, linecolor='white')
    axes[0].set_title('Size agreement')
    axes[0].set_xlabel('Clustering')
    axes[0].set_ylabel('Rules-Based')
    axes[0].tick_params(axis='x', rotation=20)
    axes[0].tick_params(axis='y', rotation=0)

    sns.heatmap(model_agree, ax=axes[1], annot=True, fmt='d', cmap='Oranges',
                linewidths=0.5, linecolor='white')
    axes[1].set_title('Model agreement')
    axes[1].set_xlabel('Clustering')
    axes[1].set_ylabel('Rules-Based')
    axes[1].tick_params(axis='x', rotation=20)
    axes[1].tick_params(axis='y', rotation=0)

    plt.tight_layout()
    plt.show()

    size_pct  = (merged['archetype_size']  == merged['rules_size']).mean()  * 100
    model_pct = (merged['archetype_model'] == merged['rules_model']).mean() * 100
    print(f'Size  agreement:  {size_pct:.1f}%')
    print(f'Model agreement:  {model_pct:.1f}%')

    disagreements = merged[
        (merged['archetype_size'] != merged['rules_size']) |
        (merged['archetype_model'] != merged['rules_model'])
    ][['practicekey', 'practicename', 'region',
       'rules_size', 'archetype_size',
       'rules_model', 'archetype_model']].head(20)

    if len(disagreements):
        print(f'\nSample of disagreements ({len(merged[(merged["archetype_size"] != merged["rules_size"]) | (merged["archetype_model"] != merged["rules_model"])]):,} total):')
        display(disagreements)
else:
    print('archetypes_rules.csv not found — run 01_rules_based.ipynb first to enable agreement analysis.')

---
## Save output

In [ ]:
out_cols = [
    'practicekey', 'practicename', 'region',
    'numberofsurgeries', 'unique_staff_ids',
    'private_income', 'nhs_income_est', 'total_income_est', 'nhs_share',
    'cluster_size_id', 'cluster_model_id',
    'archetype_size', 'archetype_model', 'archetype_clust',
]
df[out_cols].to_csv('archetypes_clustering.csv', index=False)
print(f'Saved archetypes_clustering.csv  ({len(df):,} rows)')
df[out_cols].head()